In [0]:
spark.conf.set(
    "fs.azure.account.key.batteryhealthdatalake.dfs.core.windows.net", 
    "SLUYElCah4m+RAnXDvZ9kBs1bi8/jnqVbge/BKCs+BcfcWrF9o5cmjcuW6S7/xsRzwJknh9vrYxa+AStwWjTSA=="
)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

In [0]:
from pyspark.sql import functions as F

# gold layer path
gold_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/gold/"

df_gold = f"{gold_path}daily_battery_kpis"
df_gold = spark.read.format("delta").load(df_gold)

In [0]:
df_gold.display()

In [0]:
df_gold = df_gold.dropna(subset=["voltage_stability", "peak_temp", "total_voltage_sags"])
df = df_gold.select("voltage_stability", "peak_temp", "total_voltage_sags")

# vectorize input features
assembler = VectorAssembler(inputCols=["voltage_stability", "peak_temp"], outputCol="features")
output = assembler.transform(df)

# split data
train, test = output.randomSplit([0.7, 0.3])

# train model
lr = LinearRegression(featuresCol="features", labelCol="total_voltage_sags")
lr_model = lr.fit(train)

# model prediction
prediction = lr_model.transform(test)


In [0]:
trainingSummary = lr_model.summary

print(f"RMSE: {trainingSummary.rootMeanSquaredError}")
print(f"r2: {trainingSummary.r2}")

Logistic Regression

In [0]:
from pyspark.ml.classification import LogisticRegression

df_classification = df_gold.withColumn("is_high_risk_day", F.when(F.col("total_voltage_sags") > 0, 1).otherwise(0))

df_vectorized = assembler.transform(df_classification)
train, test = df_vectorized.randomSplit([0.7, 0.3], seed=42)

# model train
log_regression_model = LogisticRegression(featuresCol="features", labelCol="is_high_risk_day")
model_fit = log_regression_model.fit(train)

# model prediction
prediction = model_fit.transform(test)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="is_high_risk_day")
print(f"Model evaluation: {evaluator.evaluate(prediction)}")